# AKANE E10-1：辞書の確かめの弱点を、手書きの一覧なしで直し、新しい問題で確かめる

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：sims/e10/PLAN_E10-1.md。

- E10で使っていない新しい問題（部分と全体・病気・地理）。正解はこのノートに入っていません。
- 辞書の鎖は **手で書いた語の一覧を使いません**（頭の名詞は文法で候補を出してNLIで選ぶ、「〜of／in〜」は全部たどってNLIで決める）。
- 弱点の直し：直し1・2（部分と全体の受け継ぎ・全体の側から見る）は材料をここで取り、計算は後で。直し3（向き）はGemmaに2択で聞きます。直し4（病名を分ける）は分けた語を辞書で引きます。
- 比べのためだけに、E10の手書きの一覧つきの鎖も同じ材料で作ります（OLD）。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（E8〜E10と同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E10-1_results/` に `E101_*.json.gz`（数個）ができます。最後のセルが一覧と大きさを出します。
5. 時間の見込み：Gemma 10〜15分、ダンプを取って読む 30〜50分（この間「GPUを使っていません」の警告が出ますが×で閉じてください）、語の一覧 10〜20分、NLIとGemmaの意味選び 90〜150分。合わせて3〜4時間。
6. 途中で切れても、もう一度「すべてのセルを実行」すれば、済んだ所は飛ばします（途中の控えをドライブに保存しています）。

In [ ]:
import os, json, time, bz2, gzip, subprocess, sys, hashlib
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E10-1_results"
os.makedirs(OUT, exist_ok=True)
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E6〜E10と同じ
GEMMA_ID = "google/gemma-2-2b-it"
DUMP_URL = "https://dumps.wikimedia.org/enwiktionary/latest/enwiktionary-latest-pages-articles.xml.bz2"
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=False)
def save_gz(obj, name):
    p = f"{OUT}/{name}.json.gz"
    with gzip.open(p, "wt", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False)
    print("保存:", name, round(os.path.getsize(p) / 1e6, 2), "MB"); return p
def load_gz(name):
    p = f"{OUT}/{name}.json.gz"
    if not os.path.exists(p): return None
    with gzip.open(p, "rt", encoding="utf-8") as f: return json.load(f)

In [ ]:
# 中身（sims/e10/e101/e101_core.py と同じ。手書きの語の一覧はありません）
"""E10-1の中身（Colabと手元の両方で使う）。手で書いた語の一覧は使わない（PLAN_E10-1.md、監査_手書き一覧.md）。
E10（e10_core.py）からの変更：
1. 頭の名詞：文法だけで候補を出し（最初の名詞のかたまりの頭と、それに付いた of の目的語。見出し形とそのままの形）、
   どれを使うかはその段のNLIの値で決める（E8nと同じ、set_genus）。PERSON_HEADS・OF_HEADS・META はやめた。
2. 一部・場所の段：「N of Y」「N in Y」は N を問わず全部たどる（LOC_HEADS はやめた）。段が成り立つかはNLIの値で決まる。
   前置詞 of・in は、問いの文の形（is a part of／is located in）から来る物。
3. ダンプから、名詞・固有名詞の意味に加えて、形容詞の意味（病名を分ける直し4で、部位の形容詞 renal などを引くためだけ）と、
   「Meronyms（部分の一覧）」「Holonyms（全体の一覧）」の語（直し2）を読む。
4. 病気の「〜の一部か」の文の器官名は、分類名から文法で作る（system_of）。手書きの部位名はやめた。
書式の読み方（DROP_T など、Wiktionaryのテンプレートの扱い）は E10 と同じ（監査のD：知識ではない）。
"""
import re

MAX_DEFS_ENT, MAX_DEFS_MID, DEPTH, MAX_LOC, MAX_CAND = 8, 8, 3, 3, 4
LOC_PREPS = ("of", "in")   # 問いの文の形（is a part of／is located in）から


# --- 頭の名詞の候補（E8n と同じ） ---
def _clean(defn):
    return re.sub(r"\([^)]*\)", " ", defn).strip()


def genus_candidates(doc, max_c=MAX_CAND):
    chunks = [c for c in doc.noun_chunks if c.root.pos_ in ("NOUN", "PROPN")]
    if not chunks:
        return []
    root = chunks[0].root
    toks = [root] + [o for ch in root.children if ch.dep_ == "prep" and ch.text.lower() == "of"
                     for o in ch.children if o.dep_ == "pobj" and o.pos_ in ("NOUN", "PROPN")]
    out = []
    for tk in toks:
        for w in (tk.lemma_.lower(), tk.text.lower()):
            if w not in out and re.fullmatch(r"[a-z][a-z\-]*", w):
                out.append(w)
    return out[:max_c]


def _np_text(tok):
    """前置詞の目的語のまとまり。固有名詞なら複合語ごと（"East Asia"）、ふつうの名詞は見出しの形。"""
    if tok.pos_ == "PROPN":
        left = [c for c in tok.lefts if c.dep_ in ("compound", "amod") and c.pos_ in ("PROPN", "ADJ") and c.text[:1].isupper()]
        i0 = min([c.i for c in left] + [tok.i])
        return " ".join(t.text for t in tok.doc[i0:tok.i + 1])
    lem = tok.lemma_.lower()
    return lem if re.fullmatch(r"[a-z][a-z\-]*", lem) else None


def loc_targets(doc):
    """「N of Y」「N in Y」の Y を全部（N を問わない）。"""
    out = []
    for tok in doc:
        if tok.dep_ != "prep" or tok.text.lower() not in LOC_PREPS:
            continue
        if tok.head.pos_ not in ("NOUN", "PROPN"):
            continue
        for o in [c for c in tok.children if c.dep_ == "pobj"]:
            for y in [o] + [c for c in o.children if c.dep_ in ("conj", "appos") and c.pos_ == o.pos_]:
                s = _np_text(y)
                if s and s not in out:
                    out.append(s)
    return out[:MAX_LOC]


def lemmas_of(doc):
    return {t.lemma_.lower() for t in doc} | {t.text.lower() for t in doc}


# --- ダンプの読み取り（E10と同じ書き換え＋形容詞・Meronyms・Holonyms） ---
DROP_T = {"lb", "lbl", "label", "q", "qual", "qualifier", "i", "gloss", "gl", "sense", "s", "defdate", "rfdef", "rfclarify",
          "senseid", "tlb", "def-date", "c", "C", "topics", "cat", "rfex", "rfquote", "nb...", "ux", "uxi", "quote-book"}
ARG2_T = {"l", "m", "link", "mention", "ll", "l-self", "m-self", "cog"}
ARG1_T = {"w", "taxlink", "taxfmt", "vern", "ws", "pedia", "non-gloss", "n-g", "ngd", "non-gloss definition", "glossary",
          "smallcaps", "taxon", "spelink", "nowrap", "small", "sub", "sup"}
SKIP_LINE = ("plural of", "alternative form of", "alternative spelling of", "obsolete form of", "misspelling of",
             "abbreviation of", "initialism of", "acronym of", "obsolete spelling of", "archaic spelling of", "eye dialect of",
             "nonstandard spelling of", "synonym of", "clipping of", "short for", "ellipsis of", "contraction of")
SKIP_T = re.compile(r"^\{\{\s*(plural of|alternative form of|alt form|alt sp|alternative spelling of|obsolete form of|"
                    r"misspelling of|abbreviation of|abbr of|initialism of|acronym of|obsolete spelling of|archaic spelling of|"
                    r"eye dialect of|nonstandard spelling of|synonym of|syn of|clipping of|short for|ellipsis of|"
                    r"contraction of|form of|inflection of|en-past of)\s*\|", re.I)
LB = re.compile(r"\{\{\s*(?:lb|lbl|label|tlb)\s*\|\s*en\s*\|([^{}]*)\}\}")


def _holo(h):
    h = h.split("/")[-1].strip()
    return re.sub(r"^the ", "", h)


def render_place(args):
    pos = [a for a in args if "=" not in a]
    if pos and pos[0] == "en":
        pos = pos[1:]
    if not pos:
        return ""
    if any("<<" in a for a in pos):
        txt = "|".join(pos)
        txt = re.sub(r"<<([^<>]*)>>", lambda m: _holo(m.group(1)) if "/" in m.group(1) else m.group(1), txt)
        return txt.replace("|", " ")
    groups, cur = [], []
    for a in pos:
        if a.strip() == ";":
            if cur: groups.append(cur)
            cur = []
        else:
            cur.append(a.strip())
    if cur: groups.append(cur)
    g = groups[0]
    ptype = g[0].replace("/", " ").strip()
    holos = [_holo(h) for h in g[1:] if h and h not in (",", "in", "of")]
    s = ("an " if ptype[:1].lower() in "aeiou" else "a ") + ptype
    return s + (" in " + ", ".join(holos) if holos else "")


def _tmpl(m):
    parts = [p.strip() for p in m.group(1).split("|")]
    name, args = parts[0].strip(), parts[1:]
    pos = [p for p in args if "=" not in p]
    if name == "place":
        return render_place(args)
    if name in DROP_T:
        return ""
    if name in ARG2_T:
        return pos[1] if len(pos) > 1 else ""
    if name in ARG1_T:
        return pos[0] if pos else ""
    return ""


def strip_wikitext(t):
    for _ in range(6):
        t2 = re.sub(r"\{\{([^{}]*)\}\}", _tmpl, t)
        if t2 == t:
            break
        t = t2
    t = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", t)
    t = re.sub(r"'{2,}|<[^>]+>", "", t)
    t = re.sub(r"\s+", " ", t).strip(" ;,.")
    return (t[:1].upper() + t[1:] + ".") if t else ""


def labels_of(raw):
    out = []
    for m in LB.finditer(raw):
        for p in m.group(1).split("|"):
            p = p.strip().lower()
            if p and "=" not in p and p not in ("_", "and", "or"):
                out.append(p)
    return out


def _senses(body, limit):
    out = []
    for line in body.splitlines():
        if not line.startswith("# "):
            continue
        raw = line[2:].strip()
        if SKIP_T.match(raw):
            continue
        d = strip_wikitext(raw)
        if len(d) > 3 and not d.lower().startswith(SKIP_LINE):
            out.append([d[:220], labels_of(raw)])
    return out[:limit]


def _linked_words(body):
    """一覧の節の語：[[X]] と、テンプレート {{名前|en|X|Y…}} の位置引数（= を含まない物）"""
    ws = re.findall(r"\[\[([^\]|#]+)(?:\|[^\]]*)?\]\]", body)
    for m in re.finditer(r"\{\{([^{}]*)\}\}", body):
        ps = [p.strip() for p in m.group(1).split("|")]
        if len(ps) > 2 and ps[1] == "en":
            ws += [p for p in ps[2:] if p and "=" not in p and "<" not in p]
    return [w.strip() for w in ws if re.fullmatch(r"[A-Za-z][A-Za-z\-' ]{0,40}", w.strip()) and not w.startswith(("Thesaurus", "Category", "Appendix"))]


def parse_page(wt):
    """英語の節から {"n": 名詞・固有名詞の意味, "a": 形容詞の意味, "mero": 部分の一覧, "holo": 全体の一覧}"""
    m = re.search(r"(?ms)^==English==\s*$(.*?)(?=^==[^=]|\Z)", wt or "")
    if not m:
        return None
    en = m.group(1)
    out = {"n": [], "a": [], "mero": [], "holo": []}
    for sec in re.finditer(r"(?ms)^(===+)\s*([^=\n]+?)\s*\1\s*$(.*?)(?=^===|\Z)", en):
        head, body = sec.group(2).strip().lower(), sec.group(3)
        if head in ("noun", "proper noun"):
            out["n"] += _senses(body, MAX_DEFS_MID)
        elif head == "adjective":
            out["a"] += _senses(body, MAX_DEFS_MID)
        elif head == "meronyms":
            out["mero"] += _linked_words(body)
        elif head == "holonyms":
            out["holo"] += _linked_words(body)
    out["n"], out["a"] = out["n"][:MAX_DEFS_MID], out["a"][:MAX_DEFS_MID]
    return out if any(out.values()) else None


TITLE_OK = re.compile(r"[A-Za-z][A-Za-z\-'.]*(?: [A-Za-z][A-Za-z\-'.]*){0,3}")


def parse_dump(fobj, log=print, every=500000):
    """WKD（名詞・固有名詞、E10と同じ形）, WKA（形容詞）, MERO, HOLO"""
    import xml.etree.ElementTree as ET
    WKD, WKA, MERO, HOLO = {}, {}, {}, {}
    n, title, ns = 0, None, None
    for ev, el in ET.iterparse(fobj, events=("end",)):
        tag = el.tag.rsplit("}", 1)[-1]
        if tag == "title":
            title = el.text
        elif tag == "ns":
            ns = el.text
        elif tag == "text":
            if ns == "0" and title and TITLE_OK.fullmatch(title) and el.text and "==English==" in el.text:
                r = parse_page(el.text)
                if r:
                    if r["n"]: WKD[title] = r["n"]
                    if r["a"]: WKA[title] = r["a"]
                    if r["mero"]: MERO[title] = sorted(set(r["mero"]))
                    if r["holo"]: HOLO[title] = sorted(set(r["holo"]))
        elif tag == "page":
            n += 1
            el.clear()
            if n % every == 0:
                log(f"ページ {n:,}  名詞のある語 {len(WKD):,}")
    return WKD, WKA, MERO, HOLO


def lookup(WKD, w):
    for v in (w, w[:1].upper() + w[1:], w[:1].lower() + w[1:], re.sub(r"^[Tt]he ", "", w)):
        if v in WKD:
            return v
    return None


# --- 語の意味の一覧（一覧なし） ---
def build_lexicon(ents_by_dom, WKD, nlp, WKA=None, adj_ok=(), log=print):
    """L[w] = [{"def","cands","loc","lem"}]。adj_ok の語は名詞の意味がなければ形容詞の意味を使う（直し4の部位の語）。
    たどる語は 頭の名詞の候補 ＋ of/in の目的語（全部）。"""
    L = {}
    def senses(w):
        if w in L:
            return L[w]
        key = lookup(WKD, w)
        ss = WKD.get(key, [])[:MAX_DEFS_MID] if key else []
        if not ss and WKA is not None and w in adj_ok:
            ka = lookup(WKA, w)
            ss = WKA.get(ka, [])[:MAX_DEFS_MID] if ka else []
        rows = []
        for d, lb in ss:
            doc = nlp(_clean(d)[:200])
            rows.append({"def": d, "cands": genus_candidates(doc), "loc": loc_targets(doc), "lem": sorted(lemmas_of(doc))})
        L[w] = rows
        return rows
    for dom, ents in ents_by_dom.items():
        front = set(ents)
        for w in ents:
            senses(w)
        for depth in range(DEPTH):
            nxt = set()
            for w in front:
                for x in L[w]:
                    for g in x["cands"] + x["loc"]:
                        if g not in L:
                            senses(g)
                        nxt.add(g)
            log(f"{dom} 段{depth + 1}: 語の合計 {len(L)}")
            front = nxt
    return L


def set_genus(L, SC):
    """SC[(w,k,g)] ＝ 一種の段のNLIの値。候補のうち辞書にあり一番大きい物を頭の名詞にする。"""
    for w, xs in L.items():
        for k, x in enumerate(xs):
            cs = [g for g in x["cands"] if g != w and L.get(g)]
            x["genus"] = max(cs, key=lambda g: SC.get((w, k, g), 0.0)) if cs else None
    return L


def next_of(L, w, k, isa_only=False):
    x = L[w][k]
    out = []
    if x.get("genus") and L.get(x["genus"]):
        out.append((x["genus"], "isa"))
    if not isa_only:
        for g in x["loc"]:
            if g != w and L.get(g) and (g, "isa") not in out:
                out.append((g, "loc"))
    return out


def chain_nodes(L, x, choice, depth=DEPTH, isa_only=False):
    """物の語 x から depth 段までの点と段 ((w,k),(g,j),種類)。途中の語の意味は choice[(w,k,g)]（Gemmaの選び）。"""
    nodes, links = set(), set()
    front = [(x, k) for k in range(len(L.get(x, [])))]
    nodes.update(front)
    for _ in range(depth):
        nxt = []
        for (w, k) in front:
            for g, typ in next_of(L, w, k, isa_only):
                j = 0 if len(L[g]) == 1 else choice.get((w, k, g), 0)
                links.add(((w, k), (g, j), typ))
                if (g, j) not in nodes:
                    nodes.add((g, j)); nxt.append((g, j))
        front = nxt
    return nodes, links


def choice_prompt(w, d, g, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Dictionary definition of \"{w}\": {d}\n\nIn this definition, which meaning of the word \"{g}\" is used?\n"
            f"{opts}\n\nAnswer with the number only.")


def premise(w, d):
    return f"\"{w}\" means: {d}"


cap = lambda s: s[:1].upper() + s[1:]
art = lambda w: ("an " if w[:1].lower() in "aeiou" else "a ") + w


def isa_hyp(w, g):
    return f"{cap(w)} is a type of {g}."


def step_hyp(w, g, typ, dom):
    if typ == "isa":
        return isa_hyp(w, g)
    return {"P": f"{cap(w)} is a part of {g}.", "G": f"{cap(w)} is located in {g}.", "D": f"{cap(w)} affects the {g}."}[dom]


def system_of(name, WKD, WKA):
    """分類名の頭の名詞（最後の語）を外し、残りの最後の語が辞書に名詞としてあり形容詞としてはない時だけ、残りを器官の名にする
    （gastrointestinal system disease → gastrointestinal system、skin disease → skin、metabolic disease → なし）。"""
    ws = name.split()
    if len(ws) < 2:
        return None
    last = ws[-2]
    return " ".join(ws[:-1]) if lookup(WKD, last) and not lookup(WKA, last) else None


def arrive_hyps(w, cls, dom, meta):
    if dom == "P":
        return [f"{cap(w)} is a part of {meta['wholes'][cls]}."]
    if dom == "G":
        return [f"{w} is located in {meta['regions'][cls]}."]
    lb = meta["labels"][cls]
    hs = [f"{cap(w)} is {lb['art']} {lb['name']}."]
    if lb.get("system"):
        hs.append(f"{cap(w)} is a part of the {lb['system']}.")
    return hs


def gemma_q(stmt):
    return f"Is the following statement true? Answer Yes or No only.\nStatement: {stmt}"   # E4〜E10と同じ聞き方


def dir_prompt(a, b):
    """直し3：向きの2択（A＝1つ目の文）。"""
    return f"Which statement is true? Answer A or B only.\nA: {a}\nB: {b}"


def split_name(name):
    """直し4：病名を最後の語（種類の候補）と残り（部位・形容詞）に分ける。2語以上の時だけ。"""
    ws = name.split()
    if len(ws) < 2:
        return None
    return ws[-1].lower(), " ".join(ws[:-1]).lower()


def disease_hyp(w):
    return f"{cap(w)} is a disease."

In [ ]:
# 比べ0のためだけ：E10の鎖（sims/e10/e10_core.py、手書きの一覧つき）を OLD という別の場所で使う
OLD_SRC = '"""E10の中身（Colabと手元の両方で使う）。E8（sims/e8/e8_core.py）からの変更点：\n1. ダンプは「名詞」に加えて「固有名詞」（地名）も読み、空白入り（4語まで）・大文字始まりの見出しも取る。\n   地名の説明文によく使われる {{place|en|city|p/Osaka|c/Japan}} を「A city in Osaka, Japan.」の形の文に直す。\n2. 鎖の段を2種類にする：\n   - 一種の段（E8と同じ、説明文の頭の名詞）\n   - 一部・場所の段（新）：説明文の中の「N of Y」「N in Y」で、N が場面ごとの決まった語の一覧に入る時だけ Y をたどる（狭いたどり方）。\n     広いたどり方（N を問わない）は部分と全体の参考だけ。\n3. 段の確からしさのNLIの言い方：一種の段は E8 と同じ「w is a type of g.」、一部・場所の段は場面ごとに\n   P「w is a part of g.」 G「w is located in g.」 D「w affects the g.」\n4. 着き（点→分類）のNLIの言い方：P「w is a part of (全体).」 G「w is located in (地域).」\n   D「w is a (分類).」と、体の部位の分類なら「w is a part of the (器官系).」の大きい方。\n意味の選び方（Gemmaに番号で選ばせる D・印のない最初の意味 B）、3段まで、一番強い道の掛け算は E8 と同じ。\n"""\nimport re\n\nMAX_DEFS_ENT, MAX_DEFS_MID, DEPTH, MAX_LOC = 8, 8, 3, 3\nBAD_LABELS = {"obsolete", "archaic", "dated", "rare", "slang", "vulgar", "offensive", "derogatory", "historical",\n              "humorous", "euphemistic", "nonstandard", "pejorative", "ethnic slur", "slur", "obsolete outside dialects",\n              "rare outside dialects", "nonce word", "jocular", "informal euphemism"}\nPERSON_HEADS = ("one who", "someone who", "somebody who", "a person who", "person who", "one that", "anyone who",\n                "a person", "someone", "somebody")\nOF_HEADS = {"kind", "type", "sort", "variety", "species", "form", "member", "one", "any", "piece", "item", "genus",\n            "class", "breed", "style", "group", "family", "unit", "strain", "subspecies", "category", "instance",\n            "example", "specimen", "individual", "portion", "quantity"}\nMETA = {"synonym", "ellipsis", "term", "name", "word", "abbreviation", "plural", "spelling", "who", "something",\n        "anything", "that", "which", "this", "one", "any", "form", "use", "sense", "variant", "misspelling", "acronym"}\n\n# 一部・場所の段をたどってよい N（結果を見る前に固定、PLAN_E10.md）\nLOC_HEADS = {\n    "P": {"part", "piece", "section", "portion", "end", "edge", "side", "front", "back", "top", "bottom", "surface",\n          "component", "organ", "segment", "bone", "muscle", "joint", "layer", "membrane", "tissue", "cavity", "area",\n          "region", "structure", "tip", "rim", "frame", "cover", "covering", "opening", "lining", "wall", "base"},\n    "G": {"city", "town", "capital", "port", "village", "municipality", "country", "nation", "state", "province",\n          "region", "island", "archipelago", "republic", "kingdom", "territory", "prefecture", "county", "district",\n          "part", "area", "emirate", "commune", "settlement", "metropolis", "suburb", "borough", "seaport", "community",\n          "governorate", "department", "division", "subdivision", "regency", "oblast", "canton", "parish", "locality"},\n    "D": {"disease", "disorder", "inflammation", "infection", "cancer", "tumor", "tumour", "condition", "carcinoma",\n          "syndrome", "illness", "cyst", "lesion", "swelling", "injury", "ailment", "malady", "failure", "deficiency",\n          "abnormality", "dysfunction", "pain", "enlargement", "narrowing", "hardening", "ulcer", "sarcoma", "lymphoma",\n          "leukemia", "leukaemia", "neoplasm", "malformation", "defect", "obstruction", "infestation"}}\nLOC_PREPS = {"of", "in", "within", "inside"}\n\n\ndef _first_head(doc):\n    chunks = list(doc.noun_chunks)\n    return chunks[0].root if chunks else None\n\n\ndef genus(defn, nlp, doc=None):\n    t = re.sub(r"\\([^)]*\\)", " ", defn).strip()\n    low = t.lower().lstrip("“\\"\' ")\n    for p in PERSON_HEADS:\n        if low.startswith(p + " ") or low == p:\n            return "person"\n    doc = doc if doc is not None else nlp(t[:200])\n    root = _first_head(doc)\n    if root is None:\n        return None\n    lem = root.lemma_.lower()\n    if lem in OF_HEADS:\n        for ch in root.children:\n            if ch.dep_ == "prep" and ch.text.lower() == "of":\n                objs = [c for c in ch.children if c.dep_ == "pobj"]\n                if objs:\n                    lem = objs[0].lemma_.lower()\n                    break\n    if lem == "being" and "human" in list(doc.noun_chunks)[0].text.lower():\n        lem = "human"\n    if lem == "specie":\n        lem = "species"\n    if lem in META or not re.fullmatch(r"[a-z][a-z\\-]*", lem):\n        return None\n    return lem\n\n\ndef _np_text(tok):\n    """前置詞の目的語のまとまり。固有名詞なら複合語ごと（"East Asia"）、ふつうの名詞は見出しの形（"lungs"→"lung"）。"""\n    if tok.pos_ == "PROPN":\n        left = [c for c in tok.lefts if c.dep_ in ("compound", "amod") and c.pos_ in ("PROPN", "ADJ") and c.text[:1].isupper()]\n        i0 = min([c.i for c in left] + [tok.i])\n        return " ".join(t.text for t in tok.doc[i0:tok.i + 1])\n    lem = tok.lemma_.lower()\n    return lem if re.fullmatch(r"[a-z][a-z\\-]*", lem) and lem not in META else None\n\n\ndef loc_targets(defn, nlp, dom, wide=False, doc=None):\n    """「N of Y」「N in Y」の Y（N が場面の一覧にある時、wide なら N を問わない）。P では「part of Y」は N を問わない。"""\n    t = re.sub(r"\\([^)]*\\)", " ", defn).strip()\n    doc = doc if doc is not None else nlp(t[:200])\n    heads = LOC_HEADS[dom]\n    out = []\n    for tok in doc:\n        if tok.dep_ != "prep" or tok.text.lower() not in LOC_PREPS:\n            continue\n        n = tok.head\n        if n.pos_ not in ("NOUN", "PROPN"):\n            continue\n        if not (wide or n.lemma_.lower() in heads):\n            continue\n        for o in [c for c in tok.children if c.dep_ == "pobj"]:\n            objs = [o] + [c for c in o.children if c.dep_ in ("conj", "appos") and c.pos_ == o.pos_]\n            for y in objs:\n                s = _np_text(y)\n                if s and s not in out:\n                    out.append(s)\n    return out[:MAX_LOC]\n\n\n# --- ダンプの読み取り（E8と同じ書き換え＋{{place}}） ---\nDROP_T = {"lb", "lbl", "label", "q", "qual", "qualifier", "i", "gloss", "gl", "sense", "s", "defdate", "rfdef", "rfclarify",\n          "senseid", "tlb", "def-date", "c", "C", "topics", "cat", "rfex", "rfquote", "nb...", "ux", "uxi", "quote-book"}\nARG2_T = {"l", "m", "link", "mention", "ll", "l-self", "m-self", "cog"}\nARG1_T = {"w", "taxlink", "taxfmt", "vern", "ws", "pedia", "non-gloss", "n-g", "ngd", "non-gloss definition", "glossary",\n          "smallcaps", "taxon", "spelink", "nowrap", "small", "sub", "sup"}\nSKIP_LINE = ("plural of", "alternative form of", "alternative spelling of", "obsolete form of", "misspelling of",\n             "abbreviation of", "initialism of", "acronym of", "obsolete spelling of", "archaic spelling of", "eye dialect of",\n             "nonstandard spelling of", "synonym of", "clipping of", "short for", "ellipsis of", "contraction of")\nSKIP_T = re.compile(r"^\\{\\{\\s*(plural of|alternative form of|alt form|alt sp|alternative spelling of|obsolete form of|"\n                    r"misspelling of|abbreviation of|abbr of|initialism of|acronym of|obsolete spelling of|archaic spelling of|"\n                    r"eye dialect of|nonstandard spelling of|synonym of|syn of|clipping of|short for|ellipsis of|"\n                    r"contraction of|form of|inflection of|en-past of)\\s*\\|", re.I)\nLB = re.compile(r"\\{\\{\\s*(?:lb|lbl|label|tlb)\\s*\\|\\s*en\\s*\\|([^{}]*)\\}\\}")\n\n\ndef _holo(h):\n    h = h.split("/")[-1].strip()\n    return re.sub(r"^the ", "", h)\n\n\ndef render_place(args):\n    """{{place|en|...}} を文に。新しい書き方（<<city>> in <<p/Osaka>>）と古い書き方（city|p/Osaka|c/Japan）の両方。"""\n    pos = [a for a in args if "=" not in a]\n    if pos and pos[0] == "en":\n        pos = pos[1:]\n    if not pos:\n        return ""\n    if any("<<" in a for a in pos):\n        txt = "|".join(pos)\n        txt = re.sub(r"<<([^<>]*)>>", lambda m: _holo(m.group(1)) if "/" in m.group(1) else m.group(1), txt)\n        return txt.replace("|", " ")\n    groups, cur = [], []\n    for a in pos:\n        if a.strip() == ";":\n            if cur: groups.append(cur)\n            cur = []\n        else:\n            cur.append(a.strip())\n    if cur: groups.append(cur)\n    g = groups[0]\n    ptype = g[0].replace("/", " ").strip()\n    holos = [_holo(h) for h in g[1:] if h and h not in (",", "in", "of")]\n    s = ("an " if ptype[:1].lower() in "aeiou" else "a ") + ptype\n    return s + (" in " + ", ".join(holos) if holos else "")\n\n\ndef _tmpl(m):\n    parts = [p.strip() for p in m.group(1).split("|")]\n    name, args = parts[0].strip(), parts[1:]\n    pos = [p for p in args if "=" not in p]\n    if name == "place":\n        return render_place(args)\n    if name in DROP_T:\n        return ""\n    if name in ARG2_T:\n        return pos[1] if len(pos) > 1 else ""\n    if name in ARG1_T:\n        return pos[0] if pos else ""\n    return ""\n\n\ndef strip_wikitext(t):\n    for _ in range(6):\n        t2 = re.sub(r"\\{\\{([^{}]*)\\}\\}", _tmpl, t)\n        if t2 == t:\n            break\n        t = t2\n    t = re.sub(r"\\[\\[(?:[^\\]|]*\\|)?([^\\]]*)\\]\\]", r"\\1", t)\n    t = re.sub(r"\'{2,}|<[^>]+>", "", t)\n    t = re.sub(r"\\s+", " ", t).strip(" ;,.")\n    return (t[:1].upper() + t[1:] + ".") if t else ""\n\n\ndef labels_of(raw):\n    out = []\n    for m in LB.finditer(raw):\n        for p in m.group(1).split("|"):\n            p = p.strip().lower()\n            if p and "=" not in p and p not in ("_", "and", "or"):\n                out.append(p)\n    return out\n\n\ndef noun_senses_from_wikitext(wt, limit=MAX_DEFS_MID):\n    """名詞と固有名詞の意味 [[説明文, [印]], ...]（ページに出る順）"""\n    m = re.search(r"(?ms)^==English==\\s*$(.*?)(?=^==[^=]|\\Z)", wt or "")\n    if not m:\n        return []\n    out = []\n    for sec in re.finditer(r"(?ms)^===+\\s*(?:Noun|Proper noun)\\s*===+\\s*$(.*?)(?=^===|\\Z)", m.group(1)):\n        for line in sec.group(1).splitlines():\n            if not line.startswith("# "):\n                continue\n            raw = line[2:].strip()\n            if SKIP_T.match(raw):\n                continue\n            d = strip_wikitext(raw)\n            if len(d) > 3 and not d.lower().startswith(SKIP_LINE):\n                out.append([d[:220], labels_of(raw)])\n    return out[:limit]\n\n\nTITLE_OK = re.compile(r"[A-Za-z][A-Za-z\\-\'.]*(?: [A-Za-z][A-Za-z\\-\'.]*){0,3}")\n\n\ndef parse_dump(fobj, log=print, every=500000):\n    import xml.etree.ElementTree as ET\n    out, n, title, ns = {}, 0, None, None\n    for ev, el in ET.iterparse(fobj, events=("end",)):\n        tag = el.tag.rsplit("}", 1)[-1]\n        if tag == "title":\n            title = el.text\n        elif tag == "ns":\n            ns = el.text\n        elif tag == "text":\n            if ns == "0" and title and TITLE_OK.fullmatch(title) and el.text and "==English==" in el.text:\n                ds = noun_senses_from_wikitext(el.text)\n                if ds:\n                    out[title] = ds\n        elif tag == "page":\n            n += 1\n            el.clear()\n            if n % every == 0:\n                log(f"ページ {n:,}  名詞のある語 {len(out):,}")\n    return out\n\n\ndef is_bad(labels):\n    return any(l in BAD_LABELS for l in labels)\n\n\ndef first_good(senses):\n    for k, s in enumerate(senses):\n        if not is_bad(s["labels"] if isinstance(s, dict) else s[1]):\n            return k\n    return 0\n\n\ndef lookup(WKD, w):\n    """見出しを探す：そのまま → 先頭だけ大文字/小文字を変えて → \'the \' を外して。"""\n    for v in (w, w[:1].upper() + w[1:], w[:1].lower() + w[1:], re.sub(r"^[Tt]he ", "", w)):\n        if v in WKD:\n            return v\n    return None\n\n\n# --- 鎖 ---\ndef build_lexicon(ents_by_dom, WKD, nlp, log=print, wide_doms=("P",)):\n    """L[w] = [{"def","labels","genus","loc":{dom: [...]}, "locw":{dom: [...]}}]。\n    ents_by_dom: {dom: [物の語]}。たどる語は場面ごとに集める（一部・場所の段の決まりが場面ごとに違うため）。"""\n    L = {}\n    def senses(w):\n        if w in L:\n            return L[w]\n        key = lookup(WKD, w)\n        ss = WKD.get(key, [])[:MAX_DEFS_MID] if key else []\n        rows = []\n        for d, lb in ss:\n            t = re.sub(r"\\([^)]*\\)", " ", d).strip()\n            doc = nlp(t[:200])\n            rows.append({"def": d, "labels": lb, "genus": genus(d, nlp, doc),\n                         "loc": {dm: loc_targets(d, nlp, dm, False, doc) for dm in "PGD"},\n                         "locw": {dm: loc_targets(d, nlp, dm, True, doc) for dm in wide_doms}})\n        L[w] = rows\n        return rows\n    for dom, ents in ents_by_dom.items():\n        front = set(ents)\n        for w in ents:\n            senses(w)\n        for depth in range(DEPTH):\n            nxt = set()\n            for w in front:\n                for x in L[w]:\n                    tg = ([x["genus"]] if x["genus"] else []) + x["loc"][dom] + (x["locw"].get(dom, []) if dom in wide_doms else [])\n                    for g in tg:\n                        if g not in L:\n                            senses(g)\n                        nxt.add(g)\n            log(f"{dom} 段{depth + 1}: 語の合計 {len(L)}")\n            front = nxt\n    return L\n\n\ndef next_of(L, w, k, dom, wide=False):\n    """(w,k) から出る段：[(g, \'isa\'|\'loc\')]"""\n    x = L[w][k]\n    out = []\n    if x["genus"] and x["genus"] != w and L.get(x["genus"]):\n        out.append((x["genus"], "isa"))\n    locs = x["locw"].get(dom, []) if wide else x["loc"][dom]\n    for g in locs:\n        if g != w and L.get(g) and (g, "isa") not in out:\n            out.append((g, "loc"))\n    return out\n\n\ndef chain_nodes(L, x, dom, method, choice=None, wide=False):\n    """物の語 x から3段までの点（語, 意味の番号）と段 ((w,k),(g,j),種類)。\n    D：途中の語の意味は choice[(w,k,g)]（Gemmaの選び）。B：印のない最初の意味。物の語は全部の意味。"""\n    nodes, links = set(), set()\n    front = [(x, k) for k in range(len(L.get(x, [])))]\n    nodes.update(front)\n    for _ in range(DEPTH):\n        nxt = []\n        for (w, k) in front:\n            for g, typ in next_of(L, w, k, dom, wide):\n                if len(L[g]) == 1:\n                    j = 0\n                elif method == "B":\n                    j = first_good(L[g])\n                else:\n                    j = (choice or {}).get((w, k, g), 0)\n                links.add(((w, k), (g, j), typ))\n                if (g, j) not in nodes:\n                    nodes.add((g, j)); nxt.append((g, j))\n        front = nxt\n    return nodes, links\n\n\ndef choice_prompt(w, d, g, cands):\n    opts = "\\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))\n    return (f"Dictionary definition of \\"{w}\\": {d}\\n\\nIn this definition, which meaning of the word \\"{g}\\" is used?\\n"\n            f"{opts}\\n\\nAnswer with the number only.")\n\n\ndef premise(w, d):\n    return f"\\"{w}\\" means: {d}"\n\n\ncap = lambda s: s[:1].upper() + s[1:]\n\n\ndef step_hyp(w, g, typ, dom):\n    if typ == "isa":\n        return f"{cap(w)} is a type of {g}."\n    return {"P": f"{cap(w)} is a part of {g}.", "G": f"{cap(w)} is located in {g}.", "D": f"{cap(w)} affects the {g}."}[dom]\n\n\ndef arrive_hyps(w, cls, dom, meta):\n    """点 w が分類 cls に「着く」かを問う文（1つか2つ）"""\n    if dom == "P":\n        return [f"{cap(w)} is a part of {meta[\'wholes\'][cls]}."]\n    if dom == "G":\n        return [f"{w} is located in {meta[\'regions\'][cls]}."]\n    lb = meta["labels"][cls]\n    hs = [f"{cap(w)} is {lb[\'art\']} {lb[\'name\']}."]\n    if lb["system"]:\n        hs.append(f"{cap(w)} is a part of the {lb[\'system\']}.")\n    return hs\n\n\ndef gemma_q(stmt):\n    return f"Is the following statement true? Answer Yes or No only.\\nStatement: {stmt}"   # E4〜E9と同じ聞き方\n'
OLD = {}
exec(OLD_SRC, OLD)
print('OLD:', sorted(k for k in OLD if k.isupper())[:8])

In [ ]:
META = json.loads(r'''{"wholes": {"architecture": "an architecture", "bacteria": "a bacteria", "beef": "a beef", "bridge": "a bridge", "bud": "a bud", "cat": "a cat", "chapel": "a chapel", "chick": "a chick", "clock": "a clock", "deer": "a deer", "dog": "a dog", "elephant": "an elephant", "embassy": "an embassy", "garage": "a garage", "highway": "a highway", "kidney": "a kidney", "lamb": "a lamb", "male": "a male", "marijuana": "a marijuana", "puppy": "a puppy", "receiver": "a receiver", "reed": "a reed", "rifle": "a rifle", "road": "a road", "salmon": "a salmon", "shark": "a shark", "sheep": "a sheep", "shook": "a shook", "tank": "a tank", "wheat": "a wheat"}, "whole_synsets": {"architecture": "architecture.n.01", "bacteria": "bacteria.n.01", "beef": "beef.n.01", "bridge": "bridge.n.01", "bud": "bud.n.01", "cat": "cat.n.01", "chapel": "chapel.n.01", "chick": "chick.n.01", "clock": "clock.n.01", "deer": "deer.n.01", "dog": "dog.n.01", "elephant": "elephant.n.01", "embassy": "embassy.n.01", "garage": "garage.n.01", "highway": "highway.n.01", "kidney": "kidney.n.01", "lamb": "lamb.n.01", "male": "male.n.01", "marijuana": "marijuana.n.01", "puppy": "puppy.n.01", "receiver": "receiver.n.01", "reed": "reed.n.01", "rifle": "rifle.n.01", "road": "road.n.01", "salmon": "salmon.n.01", "shark": "shark.n.01", "sheep": "sheep.n.01", "shook": "shook.n.01", "tank": "tank.n.01", "wheat": "wheat.n.01"}, "labels": {"cardio": {"name": "cardiovascular system disease", "art": "a"}, "resp": {"name": "respiratory system disease", "art": "a"}, "gi": {"name": "gastrointestinal system disease", "art": "a"}, "nerv": {"name": "nervous system disease", "art": "a"}, "skin": {"name": "skin disease", "art": "a"}, "musc": {"name": "musculoskeletal system disease", "art": "a"}, "urin": {"name": "urinary system disease", "art": "a"}, "endo": {"name": "endocrine system disease", "art": "an"}, "immune": {"name": "immune system disease", "art": "an"}, "blood": {"name": "blood disease", "art": "a"}, "repro": {"name": "reproductive system disease", "art": "a"}, "breast": {"name": "breast disease", "art": "a"}, "chest": {"name": "thoracic disease", "art": "a"}, "metab": {"name": "metabolic disease", "art": "a"}, "infect": {"name": "infectious disease", "art": "an"}, "tumor": {"name": "cancer or other tumor", "art": "a"}, "mental": {"name": "mental disorder", "art": "a"}, "genetic": {"name": "genetic disease", "art": "a"}, "bact": {"name": "bacterial infection", "art": "a"}, "viral": {"name": "viral infection", "art": "a"}, "fungal": {"name": "fungal infection", "art": "a"}, "parasit": {"name": "parasitic infection", "art": "a"}}, "regions": {"Central America": "Central America", "Western Asia": "Western Asia", "Eastern Asia": "Eastern Asia", "Western Europe": "Western Europe", "South-eastern Asia": "South-Eastern Asia", "Northern Africa": "Northern Africa", "Australia and New Zealand": "Australia and New Zealand", "Melanesia": "Melanesia", "Western Africa": "Western Africa", "Polynesia": "Polynesia", "Southern Africa": "Southern Africa", "Southern Europe": "Southern Europe", "Northern Europe": "Northern Europe", "Caribbean": "Caribbean", "Central Asia": "Central Asia", "Micronesia": "Micronesia", "Southern Asia": "Southern Asia", "Middle Africa": "Middle Africa", "Northern America": "Northern America", "Eastern Africa": "Eastern Africa", "Eastern Europe": "Eastern Europe", "South America": "South America"}, "date": "2026-11-11", "seed": 1}''')
ORDER = json.loads(r'''{"P": ["action", "aerial", "annex", "antenna", "anterior", "antique", "apartment", "aperture", "approach", "arch", "ash", "ash-key", "axle", "balance", "bang", "barb", "barrel", "base", "batting", "beak", "beard", "beetle", "bell", "belly", "bend", "bill", "blood", "blue-belly", "boiler", "bolt", "bottle", "bottle-tree", "boulevard", "brain", "brain-stem", "brake", "breast", "brim", "brow", "bulb", "bullet", "burger", "bypass", "canal", "cannabis", "cannon", "cap", "capacitor", "capital", "cardinal", "carpet", "carving", "case", "casing", "cathode", "ceiling", "chart", "cheat", "cheek", "chest", "chin", "choke", "chromosome", "circle", "circuit", "cleaner", "clip", "clone", "club", "coat", "cock", "comb", "cone", "contact", "cope", "coping", "corner", "cornerstone", "corolla", "corona", "corpus", "cortex", "costa", "course", "court", "courtyard", "cradle", "crest", "crossing", "crown", "curve", "dashboard", "dial", "dice", "differential", "digit", "dock", "door", "doorstep", "doorway", "driver", "duct", "ear", "edible", "electrode", "elevator", "extension", "eyeball", "eyebrow", "eyelid", "facial", "factor", "feather", "feature", "fence", "fin", "finder", "flag", "floor", "flooring", "foil", "food", "forehead", "foundation", "foyer", "frame", "funeral-residence", "fungus", "gene", "god", "goose", "graphic", "grid", "groundwork", "gum", "gun-sight", "hair", "half-moon", "hall", "hammer", "header", "heading", "heat", "heating", "hemisphere", "hen", "hind", "hinge", "hood", "hoof", "hoop", "horse-trail", "interchange", "intersection", "iris", "ivy", "jack", "jacket", "jaw", "jelly", "jersey", "jewel", "jimmy", "john", "key", "keystone", "lag", "landing", "lash", "left", "lens", "level", "lid", "lift", "lighthouse", "lip", "lobby", "lobe", "lock", "lounge", "magazine", "mast", "matrix", "mock-up", "monastery", "moss", "moustache", "movement", "mustache", "muzzle", "nail", "nasal", "nib", "nook", "nose", "nucleus", "oculus", "optic", "orb", "orbit", "pad", "palate", "pale", "pane", "panther", "part", "paw", "peak", "pearl-fish", "pelvis", "petal", "photograph", "pier", "pineapple", "placenta", "plain", "plate", "plume", "poison-berry", "portion", "portrait", "product", "punch", "pupil", "quill", "radiator", "ray", "relay", "resistance", "resistor", "retina", "rib", "rim", "ring", "roadside", "rod", "roe", "roof", "room", "rotary", "roundabout", "row", "ruin", "rump", "saddle", "saliva", "sash", "scum", "section", "segment", "serum", "shaft", "shank", "sheep-tick", "shift", "shoot", "short", "shoulder", "sights", "skeleton", "skull", "slide", "smoke", "snout", "soldier-fish", "spit", "springer", "stair", "staircase", "stairway", "stave", "step", "stigma", "stock", "storey", "story", "stretcher", "stud", "style", "sucker", "sushi", "tail", "tandem", "tap", "target", "teeth", "temple", "thick-knee", "threshold", "time-ball", "toe", "tom-tom", "tongue", "tongue-flower", "tornado", "total", "track", "tread", "trigger", "trunk", "tube", "turnaround", "turnout", "turret", "tyre", "upstairs", "vaccine", "vane", "variation", "vas", "vent-hole", "vertical", "vessel", "volleyball", "wall", "wave", "web", "weed", "weeds", "wheel", "widening", "window", "wing", "wire", "work-clothing"], "D": ["acute diarrhea", "aspiration pneumonia", "swine influenza", "Stevens-Johnson syndrome", "adult T-cell leukemia/lymphoma", "abdominal tuberculosis", "alcohol-related birth defects", "organ system cancer", "Dent disease", "vitamin metabolic disorder", "nicotine dependence", "Ogden syndrome", "sensory system cancer", "heavy chain disease", "delta chain disease", "acquired metabolic disease", "Kennedy's disease", "juvenile absence epilepsy", "stuttering", "peeling skin syndrome", "gastrointestinal allergy", "apple allergy", "plum allergy", "carp allergy", "white shrimp allergy", "snail allergy", "cardiac tuberculosis", "congenital diarrhea", "early-onset Parkinson's disease", "vegetable allergy", "mitochondrial Parkinson's disease", "vascular-type bleeding disorder", "bone remodeling disease", "Coronavirus infectious disease", "kidney cortex disease", "chronic asthma", "acute asthma", "nocturnal asthma", "subjective cognitive decline", "parsley allergy", "traumatic brain injury", "Fraser syndrome", "enhanced S-cone syndrome", "Ballard syndrome", "SHORT syndrome", "Gordon Holmes syndrome", "CHILD syndrome", "CK syndrome", "pelvic inflammatory disease", "small intestine cancer", "breast cyst", "stomach cancer", "intellectual disability", "short bowel syndrome", "separation anxiety disorder", "renal hypertension", "kidney failure", "premature menopause", "malignant hypertension", "tooth disease", "borderline personality disorder", "dependent personality disorder", "obsessive-compulsive disorder", "multiple personality disorder", "impulse control disorder", "lens disease", "trench fever", "neonatal respiratory failure", "T cell deficiency", "acute orbital inflammation", "appendix cancer", "appendix lymphoma", "rabies", "expressive language disorder", "acquired night blindness", "Chandler syndrome", "functional gastric disease", "pelvic muscle wasting", "allergic disease", "cartilage disease", "globe disease", "hepatitis A", "diarrhea", "reading disorder", "Stiff-Person syndrome", "color blindness", "secondary Parkinson disease", "nutmeg liver", "excessive tearing", "red-green color blindness", "cerebral malaria", "Masters-Allen syndrome", "frozen shoulder", "Machado-Joseph disease", "cerebral degeneration", "geographic tongue", "primary pulmonary hypertension", "FG syndrome", "Larsen syndrome", "colon carcinoma", "colon lymphoma", "alcohol use disorder", "acute chest syndrome", "childhood absence epilepsy", "epilepsy", "cannabis dependence", "skull cancer", "optic nerve disease", "placenta cancer", "communication disorder", "hepatitis D", "autoimmune hepatitis", "factor XIII deficiency", "factor XII deficiency", "hepatitis", "cervix disease", "lateral sclerosis", "West Nile fever", "multiple sclerosis", "iris disease", "tooth erosion", "alcoholic psychosis", "Gilbert syndrome", "auditory system disease", "narcissistic personality disorder", "long QT syndrome", "familial Mediterranean fever", "substance-related disorder", "spinal cord disease", "plague", "cerebral arterial disease", "anterior compartment syndrome", "heart valve disease", "bile duct disease", "skin cancer", "Alexander disease", "external ear carcinoma", "smooth muscle tumor", "retinal cancer", "Nelson syndrome", "thoracic cancer", "ear cancer", "inner ear cancer", "simple partial epilepsy", "infertility", "inhibited male orgasm", "external ear cancer", "pulmonary valve disease", "composite lymphoma", "adult lymphoma", "benign vascular tumor", "reactive arthritis", "acne", "heart lymphoma", "chest wall lymphoma", "lung lymphoma", "eye lymphoma", "mitochondrial metabolism disease", "B-cell lymphoma", "childhood leukemia", "placenta disease", "cocaine abuse", "cork-handlers' disease", "gum cancer", "upper lip cancer", "smallpox", "intermediate coronary syndrome", "Wilson disease", "Evans' syndrome", "lower gum cancer", "allergic asthma", "Sheehan syndrome", "blue drum syndrome", "peripheral vertigo", "congenital heart block", "lymph node disease", "cocaine dependence"], "G": ["Bani Yas City", "Al Karama", "The Palm Jumeirah", "Al Mizhar First", "Khulm", "Taloqan", "Eslam Qaleh", "Karukh", "Elbasan", "Berat", "Fier", "Peqin", "Luquembo", "N'zeto", "Quipungo", "Kunda dya Baze", "Lomas de Zamora", "Caleta Olivia", "Lobos", "La Falda", "Brigittenau", "Alsergrund", "Mariahilf", "Sankt Leonhard", "Caloundra", "Randwick", "Dandenong North", "Hawthorn", "Banja Luka", "Bugojno", "Doboj", "Cazin", "Savar", "Barishal", "Uttar Char Fasson", "Srimangal", "Forest", "Izegem", "Watermael-Boitsfort", "Oostkamp", "Koudougou", "Orodara", "Korsimoro", "Karankasso", "Stara Zagora", "Kazanlak", "Targovishte", "Dupnitsa", "Ngozi", "Rumonge", "Kayanza", "Bubanza", "Bassila", "Tchaourou", "Kpozoun", "Toukountouna", "Sengkurong", "Seria", "Tutong", "Liang", "Riberalta", "Bermejo", "San Ignacio de Velasco", "Tupiza", "Angra dos Reis", "Serrinha", "Nova Soure", "Campos Belos", "Tlokweng", "Kanye", "Thamaga", "Janeng", "Orsha", "Slonim", "Novogrudok", "Ivatsevichi", "Downtown Vancouver", "Courtenay", "Willowridge-Martingrove-Richview", "Clarence-Rockland", "Bukavu", "Mwene", "Djugu", "Kabare", "Bocaranga", "Batangafo", "Dekoa", "Kabo", "Owando", "Gamboma", "Mouyondzi", "Pokola", "Sankt Gallen", "Sitten", "Yverdon-les-Bains", "Wettingen", "Katiola", "Tanda", "Diourouzon", "Nzianouan", "La Serena", "Renca", "Hacienda La Calera", "Angol", "Kribi", "Tibati", "Figuil", "Makak", "Puqi", "Qiuji", "Shiqiaozi", "Chenlou", "Santa Marta", "San Gil", "Anaime", "Cota", "San Vicente de Moravia", "Quesada", "Esparza", "San Juan de Dios", "Moa", "Placetas", "Calimete", "La Sierpe", "Pilsen", "Opava", "Michle", "Bohnice", "Marzahn", "Bad Kreuznach", "Hille", "Vechelde", "Randers", "Hedensted", "Glostrup", "Middelfart", "Bonao", "Villa Francisca", "Salcedo", "Hato Mayor del Rey", "Sedrata", "Djidiouia", "Grarem", "Charef", "Ambato", "Machala", "Ibarra", "Rosa Zarate", "Tartu", "Haabersti", "Kristiine", "Viljandi", "Rivas-Vaciamadrid", "Chipiona", "Castilleja de la Cuesta", "Carlet", "Kibre Mengist", "Tulu Bolo", "Deder", "Adebaye", "East Helsinki", "Kuopio", "Herttoniemi", "Raahe", "Troyes", "Rosny-sous-Bois", "Savigny-le-Temple", "Le Pontet", "Oyem", "Bitam", "Oyam", "Makokou", "Bath", "Bicester", "Bideford", "Belle Vale", "Sekondi-Takoradi", "Kade", "Begoro", "Nalerigu", "Nema Kunku", "Lamin", "Kotu", "Tanji", "Camayenne", "Fria", "Mandiana", "Moussadou", "Le Gosier", "Petit-Bourg", "Le Moule", "Basse-Terre", "Volos", "Agios Dimitrios", "Mytilene", "Neapoli", "Zacapa", "Yepocapa", "Alotenango", "Tiquisate", "Bairro Militar", "Reino de Bandim", "Reino de Antula", "Plaque", "Whampoa", "Shui Chuen O", "Wo Che", "Mid Levels", "El Progreso", "La Lima", "Macuelizo", "Marcovia", "Karlovac", "Velika Gorica", "Bjelovar", "Liberpolis", "Croix-des-Bouquets", "Jacmel", "Verrettes", "Lenbe", "Nagykanizsa", "Baja", "Keszthely", "Dunaharaszti", "Letterkenny", "Greystones", "Balbriggan", "Lucan", "Ramat Gan", "Jaffa", "Givatayim", "Or Yehuda", "Pallichal", "Kotagiri", "Palavakkam", "Govardhan", "Basrah", "Najaf", "Al Diwaniyah", "Kufa", "Kermanshah", "Sirjan", "Hashtgerd", "Hoveyzeh", "Circoiscrizione IV", "Bacoli", "Villaricca", "Fossano", "Mandeville", "May Pen", "Ocho Rios", "Half Way Tree", "Zarqa", "Irbid", "Marj Al Hamam", "Judita", "Tochigi", "Maruobara", "Yoshioka", "Makubetsu", "Mombasa", "Eldoret", "Garissa", "Kimilili", "Balykchy", "Razzakov", "Iradan", "Tash-Kumyr", "Kampong Cham", "Pursat", "Lumphat", "Banlung", "Rajin", "Hoeyang", "Chunghwa", "Samho-rodongjagu", "Hwaseong-si", "Masan", "Gwangyang", "Tallang-dong", "Ban Khoan", "Xam Nua", "Pakxan", "Sekong", "Aley", "Jounieh", "Ghazieh", "El Hermel", "Jaffna", "Ambalangoda", "Kandana", "Hambantota", "Buchanan", "Ganta", "Harbel", "Saclepea", "Maputsoe", "Mohale's Hoek", "Mafeteng", "Thaba-Tseka", "Kaunas", "Eiguliai", "Naujininkai", "Telsiai", "Daugavpils", "Ventspils", "Valmiera", "Tukums", "Sirte", "Zuwarah", "Ubari", "Brak", "Skhirate", "Tinghir", "Ait Ourir", "Kelaat Mgouna", "Tiraspol", "Comrat", "Drochia", "Codru", "Ambanja", "Hell-Ville", "Didy", "Amboanjo", "Tetovo", "Kochani", "Lipkovo", "Zelino", "Douentza", "Finnkolo", "Kouri", "Goundam", "Kyimyindine", "Aungban", "Zee Kone", "Yesagyo", "Nalayh", "Khovd", "Tsetserleg", "Dzuunmod", "Mbera", "Tevragh Zeina", "Aleg", "Tembedgha", "Beau Bassin-Rose Hill", "Quatre Bornes", "Goodlands", "Baie du Tombeau", "Salima", "Liwonde", "Ntcheu", "Monkey Bay", "Agua Prieta", "La Barca", "Zacoalco de Torres", "Libres", "Hulu Langat", "Dungun", "Sungkai", "Kampung Tanjung Karang", "Monapo", "Gondola", "Namacurra", "Pebane", "Rundu", "Swakopmund", "Oshakati", "Ondangwa", "Agadez", "Nguigmi", "Mayahi", "Torodi", "Abeokuta", "Agege", "Nguru", "Ipoti", "Chinandega", "Matagalpa", "Mateare", "Juigalpa", "Nieuwegein", "Hoogeveen", "Ommoord", "Zeewolde", "Nesttun", "Halden", "Harstad", "Kongsberg", "Madhyapur Thimi", "Panauti", "Siraha", "Dailekh", "Porirua", "Whangarei", "Paraparaumu", "Karori", "Rustaq", "Sohar", "Sur", "Adam", "La Chorrera", "Chilibre", "Changuinola", "Sabanitas", "Talara", "Abancay", "Picsi", "Quillabamba", "Brookes Point", "Itogon", "Jaen", "Pililla", "Dera Ismail Khan", "Phalia", "Sillanwali", "Harappa", "Konin", "Nysa", "Krapkowice", "Osowa", "Caguas", "Trujillo Alto", "Barceloneta", "Aguadilla", "Hebron", "Old City", "Surif City", "Jericho", "Vila Nova de Gaia", "Arrentela", "Pontinha", "Baguim do Monte", "Fernando de la Mora", "Villa Hayes", "Coronel Oviedo", "Loma Plata", "Turda", "Lugoj", "Gherla", "Gura Humorului", "Novi Pazar", "Kikinda", "Sremska Mitrovica", "Vrbas", "Cyangugu", "Nyabitekeli", "Kayonza", "Umutara", "Madinah", "Sakakah", "King Faisal Military City", "Afif", "Kassala", "El Geneina Fort", "Kadugli", "Al Hasaheisa", "Vasastaden", "Tumba", "Sundbyberg", "Gustavsberg", "Kangkar", "Senja", "Telok Blangah", "Tiong Bahru Estate", "Maribor", "Kranj", "Novo Mesto", "Trbovlje", "Trnava", "Michalovce", "Bardejov", "Malacky", "Kenema", "Port Loko", "Lunsar", "Goderich", "Kolda", "Mboro", "Kayar", "Kbombole", "Berbera", "Balanbale", "Dhabad", "Ceerigaabo", "Pontbuiten", "Rainville", "Lelydorp", "Tammenga", "Malakal", "Wau", "Aweil", "Kafia Kingi", "Santa Tecla", "Delgado", "San Rafael Oriente", "Sonzacate", "Homs", "Al-Tabqa", "Al Qaryatayn", "Tallkalakh", "Kelo", "Doba", "Pala", "Massaguet", "Bassar", "Sotouboua", "Bafilo", "Tabligbo", "Suan Luang", "Bangkok Yai", "Pran Buri", "Bang Bon Nuea", "Boshkengash", "Norak", "Istiqlol", "Vakhsh", "Balkanabat", "Bayramaly", "Yylanly", "Gubadag", "Menzel Bourguiba", "El Kef", "Jendouba", "Borj Cedria", "Arima", "Diego Martin", "Point Fortin", "Tunapuna", "Mbuguni", "Namanyere", "Ngudu", "Kibara", "Kryukiv", "Debaltseve", "Korostyshiv", "Bakhmach", "Butaleja", "Anaka", "Lwakyomba", "Iryango", "The Villages", "Southglenn", "Wheat Ridge", "Carteret", "Fray Bentos", "Treinta y Tres", "Carmelo", "Young", "Kogon Shahri", "Kitob", "Shofirkon Shahri", "Chelak", "Ciudad Guayana", "La Victoria", "Carora", "La Azulita", "Haiphong", "Phu Quoc", "Lai Vung", "Mui Ne", "Al Bayda", "Ataq", "Ma'rib", "Mocha", "Centurion", "Kroonstad", "Harrismith", "Burgersdorp", "Mufulira", "Luwingu", "Manyinga", "Chiengi", "Chegutu", "Chiredzi", "Gokwe", "Gwanda"]}''')
CORDER = json.loads(r'''{"P": ["architecture", "bacteria", "beef", "bridge", "bud", "cat", "chapel", "chick", "clock", "deer", "dog", "elephant", "embassy", "garage", "highway", "kidney", "lamb", "male", "marijuana", "puppy", "receiver", "reed", "rifle", "road", "salmon", "shark", "sheep", "shook", "tank", "wheat"], "D": ["cardio", "resp", "gi", "nerv", "skin", "musc", "urin", "endo", "immune", "blood", "repro", "breast", "chest", "metab", "infect", "tumor", "mental", "genetic", "bact", "viral", "fungal", "parasit"], "G": ["Australia and New Zealand", "Caribbean", "Central America", "Central Asia", "Eastern Africa", "Eastern Asia", "Eastern Europe", "Melanesia", "Micronesia", "Middle Africa", "Northern Africa", "Northern America", "Northern Europe", "Polynesia", "South America", "South-eastern Asia", "Southern Africa", "Southern Asia", "Southern Europe", "Western Africa", "Western Asia", "Western Europe"]}''')
def remake(order, corder, meta):
    """ノートの中で主張の文を作り直す（e101_data.py と同じ書き方）。"""
    cap = lambda s: s[:1].upper() + s[1:]
    art = lambda w: ("an " if w[:1].lower() in "aeiou" else "a ") + w
    out = []
    for x in order["P"]:
        for W in corder["P"]:
            ph = meta["wholes"][W]
            out.append(["P", x, W, f"{cap(x)} is a part of {ph}.", f"{cap(ph.split(' ', 1)[1])} is a part of {art(x)}."])
    for x in order["D"]:
        for l in corder["D"]:
            lb = meta["labels"][l]
            out.append(["D", x, l, f"{cap(x)} is {lb['art']} {lb['name']}.", None])
    for x in order["G"]:
        for g in corder["G"]:
            out.append(["G", x, g, f"{x} is located in {meta['regions'][g]}.", None])
    return out
ITEMS = remake(ORDER, CORDER, META)
assert hashlib.md5(json.dumps(ITEMS).encode()).hexdigest()[:10] == 'bb5538d2f0', '主張の作り直しが合わない'
ENTS = {d: sorted({it[1] for it in ITEMS if it[0] == d}) for d in 'PDG'}
CLASSES = {d: sorted({it[2] for it in ITEMS if it[0] == d}) for d in 'PDG'}
WHOLES = sorted(META['wholes'])
print(len(ITEMS), '主張', {d: (len(ENTS[d]), len(CLASSES[d])) for d in 'PDG'}, '確認の印:', hashlib.md5(json.dumps(ITEMS).encode()).hexdigest()[:10])

## 1. Gemmaに全部の主張を「はい／いいえ」で聞き、部分と全体は向きを2択でも聞く（GPU、10〜15分）

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
def last_logits(qs, a, b, B=64, log_every=100):
    # 各問いの最後の位置で、語a と語b の値の差だけを残す（全部の語の値をGPUに溜めるとメモリが足りない）
    out = []; t = time.time()
    for i in range(0, len(qs), B):
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs[i:i + B]]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            r = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
            out.extend((r[:, a] - r[:, b]).cpu().numpy().round(3).tolist()); del r
        if (i // B) % log_every == 0: print(f"{i}/{len(qs)}  {time.time() - t:.0f}秒")
    return out
G = load_gz("E101_gemma")
if G is None:
    YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
    Q = [gemma_q(it[3]) for it in ITEMS] + [gemma_q(it[4]) for it in ITEMS if it[4]]
    lg = [float(v) for v in last_logits(Q, YES, NO)]
    n = len(ITEMS); rev = iter(lg[n:])
    G = {"logit": lg[:n], "rev": [next(rev) if it[4] else None for it in ITEMS], "model": GEMMA_ID}
    save_gz(G, "E101_gemma")
DIR = load_gz("E101_direction")
if DIR is None:
    A_ = tok.encode("A", add_special_tokens=False)[0]; B_ = tok.encode("B", add_special_tokens=False)[0]
    PI = [i for i, it in enumerate(ITEMS) if it[0] == "P"]
    Q = [dir_prompt(ITEMS[i][3], ITEMS[i][4]) for i in PI] + [dir_prompt(ITEMS[i][4], ITEMS[i][3]) for i in PI]
    d = [float(v) for v in last_logits(Q, A_, B_)]
    n = len(PI)
    DIR = {"idx": PI, "ab": d[:n], "ba": d[n:]}   # ab：A＝「x は W の一部」、ba：B＝「x は W の一部」
    save_gz(DIR, "E101_direction")
lg = np.array(G["logit"])
for d in "PDG":
    m = np.array([it[0] == d for it in ITEMS]); print(d, "はいの割合", round(float((lg[m] > 0).mean()), 3))

## 2. Wiktionaryのダンプを取って読む（名詞・固有名詞・形容詞・部分の一覧・全体の一覧、30〜50分、GPUは使わない）

In [ ]:
wk = load_gz("E101_wiktionary")
if wk is not None:
    WKD, WKA, MERO, HOLO = wk["n"], wk["a"], wk["mero"], wk["holo"]; print("ドライブから読み込み:", len(WKD))
else:
    dump = "/content/enwiktionary.xml.bz2"
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", dump, DUMP_URL], check=True)
    print("取れた大きさ:", round(os.path.getsize(dump) / 1e9, 2), "GB")
    t = time.time()
    with bz2.open(dump, "rb") as f:
        WKD, WKA, MERO, HOLO = parse_dump(f)
    print("読み終わり:", len(WKD), "語", f"{time.time() - t:.0f}秒")
    save_gz({"n": WKD, "a": WKA, "mero": MERO, "holo": HOLO, "date": time.strftime("%Y-%m-%d")}, "E101_wiktionary")
print("名詞", len(WKD), "形容詞", len(WKA), "部分の一覧", len(MERO), "全体の一覧", len(HOLO))
for d in "PDG":
    print(d, "説明文のある物の割合:", round(float(np.mean([lookup(WKD, w) is not None for w in ENTS[d]])), 3))
# 病気の「〜の一部か」の器官名を分類名から作る（手書きの部位名なし）
for l, lb in META["labels"].items():
    lb["system"] = system_of(lb["name"], WKD, WKA)
print({l: lb["system"] for l, lb in META["labels"].items()})
# 直し4：辞書にない病名を、最後の語（種類の候補）と残り（部位・形容詞）に分ける
SPLIT = {}
for x in ENTS["D"]:
    if lookup(WKD, x) is None and split_name(x):
        kind, rest = split_name(x)
        SPLIT[x] = {"kind": kind, "rest": rest, "rest_last": rest.split()[-1]}
print("辞書にない病名:", sum(lookup(WKD, x) is None for x in ENTS["D"]), " 分けられた:", len(SPLIT))

## 3. 語の意味の一覧（一覧なし・E10の手書きつきの両方、10〜20分）

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner"])
D_EXTRA = sorted({v[k] for v in SPLIT.values() for k in ("kind", "rest", "rest_last")})
ADJ_OK = {v[k] for v in SPLIT.values() for k in ("rest", "rest_last")}
LX = load_gz("E101_lexicon")
if LX is None:
    t = time.time()
    L = build_lexicon({"P": ENTS["P"] + WHOLES, "D": ENTS["D"] + D_EXTRA, "G": ENTS["G"]}, WKD, nlp, WKA, ADJ_OK)
    LO = OLD["build_lexicon"](ENTS, WKD, nlp, wide_doms=())
    print("語:", len(L), "（E10の手書きつき:", len(LO), "）", f"{time.time() - t:.0f}秒")
    save_gz({"L": L, "LO": LO}, "E101_lexicon")
else:
    L, LO = LX["L"], LX["LO"]
PAIRS = sorted({(w, k, g) for w in L for k, x in enumerate(L[w]) for g in x["cands"] if g != w and L.get(g)})
print("頭の名詞の候補の組:", len(PAIRS))

## 4. NLIで頭の名詞を選び、Gemmaに途中の語の意味を選ばせる（GPU、30〜50分、途中の控えあり）

In [ ]:
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID, torch_dtype=torch.float16).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORD = [lab["entailment"], lab["contradiction"]]
def nli_raw(P, H, B=128):
    out = np.zeros((len(P), 2), np.float32)
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORD]
        out[i:i + len(pr)] = pr.cpu().numpy()
    return out
def nli_probs(P, H, ck, CH_N=50000):
    """控え（ck_*.npy）は入力の組の md5 が合う時だけ使う"""
    out = np.zeros((len(P), 2), np.float32); t = time.time()
    for s in range(0, len(P), CH_N):
        key = hashlib.md5("\n".join(p + "\t" + h for p, h in zip(P[s:s + CH_N], H[s:s + CH_N])).encode()).hexdigest()[:12]
        p = f"{OUT}/{ck}_{s}_{key}.npy"
        if os.path.exists(p):
            out[s:s + CH_N] = np.load(p); continue
        r = nli_raw(P[s:s + CH_N], H[s:s + CH_N]); np.save(p, r); out[s:s + len(r)] = r
        print(f"{ck} {s + len(r)}/{len(P)}  {time.time() - t:.0f}秒")
    return out
SP = nli_probs([premise(w, L[w][k]["def"]) for w, k, g in PAIRS], [isa_hyp(w, g) for w, k, g in PAIRS], "ck_pairs")
SC = {p: float(s[0]) for p, s in zip(PAIRS, SP)}
set_genus(L, SC)
# 直し4：種類の候補の語が「病気」と言えるか（その語の説明文 ⇒ "{語} is a disease."）
KIND = sorted({v["kind"] for v in SPLIT.values()})
KP = [(w, k) for w in KIND for k in range(len(L.get(w, [])))]
KS = nli_probs([premise(w, L[w][k]["def"]) for w, k in KP], [disease_hyp(w) for w, k in KP], "ck_kind")
KIND_SC = {w: 0.0 for w in KIND}
for (w, k), s in zip(KP, KS): KIND_SC[w] = max(KIND_SC[w], float(s[0]))
print("種類の語と言えた:", sum(v > 0.5 for v in KIND_SC.values()), "/", len(KIND))
# 意味を選ぶ必要のある段（一覧なしの鎖と、E10の手書きつきの鎖の両方）
NEED = set()
def walk(starts, nxt_fn, LL, depth):
    for x in starts:
        front, seen = [(x, k) for k in range(len(LL.get(x, [])))], set()
        for _ in range(depth):
            nxt = []
            for (w, k) in front:
                for g, typ in nxt_fn(w, k):
                    if len(LL[g]) > 1: NEED.add((w, k, g))
                    for j in range(len(LL[g])):
                        if (g, j) not in seen: seen.add((g, j)); nxt.append((g, j))
            front = nxt
walk(ENTS["P"], lambda w, k: next_of(L, w, k), L, DEPTH)
walk(WHOLES, lambda w, k: next_of(L, w, k, True), L, 2)
walk(ENTS["D"] + D_EXTRA, lambda w, k: next_of(L, w, k), L, DEPTH)
walk(ENTS["G"], lambda w, k: next_of(L, w, k), L, DEPTH)
for d in "PDG":
    walk(ENTS[d], lambda w, k, d=d: OLD["next_of"](LO, w, k, d, False), LO, DEPTH)
NEED = sorted(NEED)
CH = load_gz("E101_choice") or []
CHOICE = {(w, k, g): v for w, k, g, v in CH}
todo = [x for x in NEED if x not in CHOICE]
print("意味を選ぶ必要のある段:", len(NEED), " 済み:", len(CHOICE), " 残り:", len(todo))
if todo:
    NUM = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(1, MAX_DEFS_MID + 1)]
    t = time.time(); B = 16
    for i in range(0, len(todo), B):
        batch = todo[i:i + B]
        LL = [L if (w in L and k < len(L[w]) and g in L) else LO for w, k, g in batch]
        qs = [choice_prompt(w, ll[w][k]["def"], g, [x["def"] for x in ll[g]]) for (w, k, g), ll in zip(batch, LL)]
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        for (w, k, g), row, ll in zip(batch, last, LL):
            CHOICE[(w, k, g)] = int(row[NUM[:len(ll[g])]].argmax())
        if (i // B) % 200 == 0:
            print(f"{i}/{len(todo)}  {time.time() - t:.0f}秒")
            save_gz([[w, k, g, v] for (w, k, g), v in CHOICE.items()], "E101_choice")
    save_gz([[w, k, g, v] for (w, k, g), v in CHOICE.items()], "E101_choice")
print("最初の意味以外を選んだ割合:", round(float(np.mean([v != 0 for v in CHOICE.values()])), 3))

## 5. 鎖を作り、NLIで段と着きを判定する（GPU、60〜100分、途中の控えあり）

In [ ]:
CHAINS = {}
for d in "PDG":
    starts = ENTS[d] + (D_EXTRA if d == "D" else [])
    for x in starts:
        CHAINS[(d, "N", x)] = chain_nodes(L, x, CHOICE)
    for x in ENTS[d]:
        CHAINS[(d, "D", x)] = OLD["chain_nodes"](LO, x, d, "D", CHOICE, False)
WCH = {W: chain_nodes(L, W, CHOICE, depth=2, isa_only=True) for W in WHOLES}   # 直し1：全体の上の種類
STEPS = sorted({(d, tag, a[0], a[1], b[0], typ) for (d, tag, x), (nd, lk) in CHAINS.items() for a, b, typ in lk if not (tag == "N" and typ == "isa")})
LEX = lambda tag: L if tag == "N" else LO
r = nli_probs([premise(w, LEX(tag)[w][k]["def"]) for d, tag, w, k, g, typ in STEPS],
              [step_hyp(w, g, typ, d) if tag == "N" else OLD["step_hyp"](w, g, typ, d) for d, tag, w, k, g, typ in STEPS], "ck_steps")
ST = [[d, tag, w, k, g, typ, round(float(p[0]), 4)] for (d, tag, w, k, g, typ), p in zip(STEPS, r)]
# 一覧なしの一種の段は、頭の名詞を選んだ時のNLIの値をそのまま使う
for (d, tag, x), (nd, lk) in CHAINS.items():
    if tag == "N":
        for a, b, typ in lk:
            if typ == "isa": ST.append([d, "N", a[0], a[1], b[0], "isa", round(SC.get((a[0], a[1], b[0]), 0.0), 4)])
for W, (nd, lk) in WCH.items():
    for a, b, typ in lk: ST.append(["W", "N", a[0], a[1], b[0], "isa", round(SC.get((a[0], a[1], b[0]), 0.0), 4)])
ST = [list(t) for t in sorted({tuple(s) for s in ST})]
save_gz(ST, "E101_steps")
# 着き：場面ごとに、一覧なし・手書きつきの両方の点。どちらも同じダンプの同じ説明文なので、(語, 意味の番号) が同じなら同じ点
def def_of(w, k):
    return L[w][k]["def"] if w in L and k < len(L[w]) else LO[w][k]["def"]
NODES = {d: sorted({(n[0], n[1]) for (dd, tag, x), (nd, lk) in CHAINS.items() if dd == d for n in nd}) for d in "PDG"}
ARR = {}
for d in "PDG":
    nh = 2 if d == "D" else 1
    nodes = NODES[d]
    P_, H_ = [], []
    for w, k in nodes:
        for c in CLASSES[d]:
            hs = arrive_hyps(w, c, d, META); hs = hs + [hs[0]] * (nh - len(hs))
            for h in hs: P_.append(premise(w, def_of(w, k))); H_.append(h)
    ARR[d] = nli_probs(P_, H_, f"ck_arrive_{d}")[:, 0].reshape(len(nodes), len(CLASSES[d]), nh)
    print(d, "着き 済み", len(nodes), "点")

## 6. 結果を小さく分けて保存（どれも10MBより小さく）

In [ ]:
used = sorted({n[0] for d in "PDG" for n in NODES[d]} | set(WHOLES) | {n[0] for W, (nd, lk) in WCH.items() for n in nd})
LU = {w: [{"def": x["def"], "genus": x.get("genus"), "loc": x["loc"], "cands": x["cands"]} for x in L[w]] for w in used if w in L}
LOU = {w: [{"def": x["def"], "genus": x["genus"], "loc": x["loc"]} for x in LO[w]] for w in used if w in LO}
WLEM = {W: [x["lem"] for x in L.get(W, [])] for W in WHOLES}
rel = set(ENTS["P"]) | set(WHOLES)
save_gz({"meta": {"nli": NLI_ID, "gemma": GEMMA_ID, "dump": DUMP_URL, "date": time.strftime("%Y-%m-%d"),
                  "items_md5": hashlib.md5(json.dumps(ITEMS).encode()).hexdigest()[:10], "labels": META["labels"]},
         "L": LU, "LO": LOU, "whole_lemmas": WLEM,
         "mero": {w: MERO[k] for w in rel for k in [lookup(MERO, w)] if k},
         "holo": {w: HOLO[k] for w in rel for k in [lookup(HOLO, w)] if k},
         "split": SPLIT, "kind_disease": KIND_SC,
         "whole_chains": {W: [[list(n) for n in nd], [[list(a), list(b), t] for a, b, t in lk]] for W, (nd, lk) in WCH.items()}}, "E101_L_used")
save_gz([[w, k, g, v] for (w, k, g), v in CHOICE.items()], "E101_choice")
for d in "PDG":
    ch = {f"{tag}|{x}": [[list(n) for n in nd], [[list(a), list(b), t] for a, b, t in lk]]
          for (dd, tag, x), (nd, lk) in CHAINS.items() if dd == d}
    save_gz({"chains": ch, "nodes": [list(n) for n in NODES[d]], "classes": CLASSES[d],
             "arrive": np.round(ARR[d], 3).tolist()}, f"E101_res_{d}")
print("\nドライブの E10-1_results の中身:")
for f in sorted(os.listdir(OUT)):
    if f.endswith(".gz"): print(f, round(os.path.getsize(f"{OUT}/{f}") / 1e6, 2), "MB")